# 한국 지수 수집 + 분석 (KRX 우회 로더 통합)

**korea_index_analysis_v4**

`kr_price_source_diagnostic_v1` 의 검증된 다중소스 로더 위에 v3 분석 모듈을 결합한 통합본입니다.

## v3 대비 변경점

| 항목 | v3 | v4 |
|---|---|---|
| 데이터 소스 | FDR 단독 (KRX 경유 → **LOGOUT 차단**) | naver → yahoo → fdr → fdr_etf 자동 폴백 |
| 휩쏘 제거 | `smooth_regime` (**사후 평활 = 미래참조**) | `confirm_signal` (확인지연, **과거만 사용**) |
| MS 국면 | smoothed 확률만 | smoothed(서술용) + **filtered(인과)** 병행 |

## 진단으로 확인된 사항 (2026-07-29 기준)

- KRX 데이터포털은 헤더 없이 403, 브라우저 헤더로도 400 `LOGOUT` → **fdr 지수·pykrx 모두 사용 불가**
- naver 정상 (2005년~현재), yahoo와 종가 상대오차 **0.0000%**
- `fdr_etf`(KODEX200)는 지수가 아닌 **ETF 가격**이라 레벨 스케일이 다릅니다.
  최후 폴백으로만 두고, 사용될 경우 수익률 기반 지표만 유효합니다.

## 국면 라벨 사용 시 주의

| 컬럼 | 인과성 | 용도 |
|---|---|---|
| `regime_ms` (smoothed) | ✗ 전표본 조건부 | 과거 국면 **서술·기술통계** 전용 |
| `regime_ms_filt` (filtered) | △ 파라미터만 전표본 | 실시간 판정에 근접 |
| `regime_rule_conf` | ✓ 완전 인과 | 실시간 판정 |

**백테스트에는 `regime_ms` 를 쓰지 마십시오.** 워크포워드 백테스트는 `korea_regime_strategy_backtest_v1` 을 사용하십시오.


## 1. Import

In [1]:
import os
import sys
import ast
import json
import time
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Union   # Python 3.9

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 240)

try:
    import FinanceDataReader as fdr
    HAS_FDR = True
except ImportError:
    HAS_FDR = False

try:
    import yfinance as yf
    HAS_YF = True
except ImportError:
    HAS_YF = False

try:
    from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

print('FinanceDataReader :', 'OK' if HAS_FDR else 'N/A')
print('yfinance          :', 'OK' if HAS_YF else 'N/A')
print('statsmodels       :', 'OK' if HAS_STATSMODELS else 'N/A -> 국면전환은 규칙기반만 수행')

FinanceDataReader : OK
yfinance          : OK
statsmodels       : OK


## 2. 사용자 입력 변수  ← **이 셀만 수정**

In [2]:
# ============================================================================
# [A] 데이터 수집
# ============================================================================
INDEX_CODES = {                                        # 수집할 지수 {코드: 이름}
    'KS11': 'KOSPI',                                   # 코스피
    'KQ11': 'KOSDAQ',                                  # 코스닥
}

START_DATE = '2005-01-01'                              # 수집 시작일
END_DATE   = datetime.today().strftime('%Y-%m-%d')     # 수집 종료일

# 소스 우선순위. 진단 결과 naver 정상 / fdr 지수·pykrx 는 KRX 차단으로 실패
SOURCE_PRIORITY = ['naver', 'yahoo', 'fdr', 'fdr_etf']
TIMEOUT       = 15                                     # HTTP 타임아웃(초)
REQUEST_SLEEP = 0.3                                    # 요청 간 대기(초)

# ============================================================================
# [B] 변동성
# ============================================================================
VOL_WINDOWS       = [20, 60, 120]                      # 실현변동성 윈도우(거래일)
VOL_ANNUALIZE     = 252                                # 연율화 계수
EWMA_LAMBDA       = 0.94                               # EWMA 감쇠계수(RiskMetrics)
VOL_RANGE_WINDOW  = 20                                 # Parkinson / Garman-Klass 평균 윈도우
VOL_PCTILE_WINDOW = 756                                # 변동성 백분위 기간(약 3년)

# ============================================================================
# [C] MDD
# ============================================================================
DD_TOP_N         = 10                                  # 상위 낙폭 국면 출력 개수
DD_MIN_DEPTH_PCT = -5.0                                # 집계할 최소 낙폭 깊이(%)

# ============================================================================
# [D] 이격도
# ============================================================================
MA_WINDOWS         = [5, 20, 60, 120, 200]             # 이동평균 윈도우
DISPARITY_WINDOWS  = [5, 20, 60, 120]                  # 이격도 산출 윈도우
DISPARITY_Z_BASE   = 20                                # z-score 기준 윈도우
DISPARITY_Z_WINDOW = 252                               # z-score 롤링 기간

# ============================================================================
# [E] 국면전환
# ============================================================================
RUN_MARKOV     = True                                  # Markov Switching 실행 여부
MS_K_REGIMES   = 2                                     # 국면 수
MS_MIN_OBS     = 500                                   # 추정 최소 관측치
MS_MAXITER     = 200                                   # EM 최대 반복

REGIME_TREND_MA      = 200                             # 규칙기반 추세 기준 MA
REGIME_VOL_PCTILE_TH = 0.70                            # 고변동 판정 백분위 임계
CONFIRM_DAYS         = 3                               # 확인지연 일수(인과적 휩쏘 억제). 1이면 미적용

# ============================================================================
# [F] 저장
# ============================================================================
SAVE_CSV = True                                        # CSV 저장 여부
SAVE_DB  = False                                       # MySQL(investar) 저장 여부

TABLE_INDEX     = 'korea_index_daily'                  # 지수 실적 테이블
TABLE_ANALYTICS = 'korea_index_analytics_daily'        # 분석 파생 테이블
TABLE_DRAWDOWN  = 'korea_index_drawdown_episode'       # 낙폭 국면 테이블
DB_CHUNK        = 1000
RECREATE_ANALYTICS_TABLE = False                       # 파라미터 변경으로 컬럼 구성이 바뀌면 True 1회

PLOT = False                                           # 시각화 실행 여부

## 3. 정합성 체크 & 경로

In [3]:
_add = [w for w in DISPARITY_WINDOWS if w not in MA_WINDOWS]
if _add:
    MA_WINDOWS = sorted(set(MA_WINDOWS) | set(_add))
    print('[FIX] MA_WINDOWS =', MA_WINDOWS)
if REGIME_TREND_MA not in MA_WINDOWS:
    MA_WINDOWS = sorted(set(MA_WINDOWS) | {REGIME_TREND_MA})
    print('[FIX] MA_WINDOWS =', MA_WINDOWS)
if DISPARITY_Z_BASE not in DISPARITY_WINDOWS:
    raise ValueError('DISPARITY_Z_BASE 는 DISPARITY_WINDOWS 안에 있어야 합니다.')

CANDIDATE_ROOTS = [
    r'C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast',
    r'C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy',
    os.getcwd(), os.path.dirname(os.getcwd()),
]


def resolve_base_dir(candidates: List[str]) -> str:
    for c in candidates:
        try:
            if c and os.path.isdir(os.path.join(c, 'DATA')):
                return c
        except Exception:
            continue
    return os.getcwd()


BASE_DIR = resolve_base_dir(CANDIDATE_ROOTS)
DATA_DIR = os.path.join(BASE_DIR, 'DATA')
OUT_DIR = os.path.join(DATA_DIR, 'market_data')
os.makedirs(OUT_DIR, exist_ok=True)
if DATA_DIR not in sys.path:
    sys.path.append(DATA_DIR)
print('OUT_DIR :', OUT_DIR)

OUT_DIR : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\market_data


## 4. 다중소스 로더

진단 노트북에서 검증된 로더입니다. `naver` 는 KRX를 경유하지 않아 LOGOUT 차단의 영향을 받지 않습니다.


In [4]:
STD_COLS = ['date', 'open', 'high', 'low', 'close', 'volume']

HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'),
    'Accept': 'application/json, text/plain, */*',
}
NAVER_INDEX_MAP = {'KS11': 'KOSPI', 'KQ11': 'KOSDAQ', 'KS200': 'KPI200'}
YAHOO_INDEX_MAP = {'KS11': '^KS11', 'KQ11': '^KQ11', 'KS200': '^KS200'}
ETF_PROXY_MAP   = {'KS11': '069500', 'KQ11': '229200', 'KS200': '069500'}


def _standardize(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    if 'date' not in df.columns:
        df = df.rename(columns={df.columns[0]: 'date'})
    for c in STD_COLS:
        if c not in df.columns:
            df[c] = np.nan
    df['date'] = pd.to_datetime(df['date'])
    for c in ['open', 'high', 'low', 'close', 'volume']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return (df[STD_COLS].dropna(subset=['date', 'close'])
            .drop_duplicates(subset=['date'], keep='last')
            .sort_values('date').reset_index(drop=True))


def parse_naver_sise(text: str) -> pd.DataFrame:
    """네이버 siseJson 응답 파서. 표준 JSON이 아닌 단일따옴표 배열로 오는 경우가 있음"""
    s = text.strip()
    if not s:
        raise ValueError('빈 응답')
    rows = None
    for how in ('json', 'quote_swap', 'literal_eval'):
        try:
            if how == 'json':
                rows = json.loads(s)
            elif how == 'quote_swap':
                rows = json.loads(s.replace("'", '"'))
            else:
                rows = ast.literal_eval(s)
            break
        except Exception:
            continue
    if not rows or len(rows) < 2:
        raise ValueError('파싱 실패 또는 데이터 없음 : {}'.format(s[:120]))

    header = [str(h).strip() for h in rows[0]]
    df = pd.DataFrame(rows[1:], columns=header)
    ren = {'날짜': 'date', '시가': 'open', '고가': 'high', '저가': 'low',
           '종가': 'close', '거래량': 'volume', '외국인소진율': 'foreign_exhaustion'}
    df = df.rename(columns={c: ren.get(c, c) for c in df.columns})
    df['date'] = pd.to_datetime(df['date'].astype(str).str.strip(),
                                format='%Y%m%d', errors='coerce')
    return df


def load_naver(symbol: str, start: str, end: str) -> pd.DataFrame:
    sym = NAVER_INDEX_MAP.get(symbol, symbol)
    url = ('https://api.finance.naver.com/siseJson.naver'
           '?symbol={}&requestType=1&startTime={}&endTime={}&timeframe=day').format(
        sym, pd.to_datetime(start).strftime('%Y%m%d'), pd.to_datetime(end).strftime('%Y%m%d'))
    r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return _standardize(parse_naver_sise(r.text))


def load_fdr(symbol: str, start: str, end: str) -> pd.DataFrame:
    if not HAS_FDR:
        raise RuntimeError('FinanceDataReader 미설치')
    return _standardize(fdr.DataReader(symbol, start, end).reset_index())


def load_fdr_etf(symbol: str, start: str, end: str) -> pd.DataFrame:
    """최후 폴백. 지수가 아닌 ETF 가격이므로 레벨 비교 불가, 수익률 지표만 유효"""
    if not HAS_FDR:
        raise RuntimeError('FinanceDataReader 미설치')
    proxy = ETF_PROXY_MAP.get(symbol)
    if proxy is None:
        raise ValueError('ETF 프록시 미정의 : {}'.format(symbol))
    return _standardize(fdr.DataReader(proxy, start, end).reset_index())


def load_yahoo(symbol: str, start: str, end: str) -> pd.DataFrame:
    if not HAS_YF:
        raise RuntimeError('yfinance 미설치')
    df = yf.download(YAHOO_INDEX_MAP.get(symbol, symbol), start=start, end=end,
                     progress=False, auto_adjust=False)
    if df is None or len(df) == 0:
        raise ValueError('빈 응답')
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return _standardize(df.reset_index())


LOADERS = {'naver': load_naver, 'yahoo': load_yahoo,
           'fdr': load_fdr, 'fdr_etf': load_fdr_etf}


def load_price_multi(symbol: str, start: str, end: str,
                     priority: Optional[List[str]] = None,
                     verbose: bool = True) -> Tuple[pd.DataFrame, str]:
    """우선순위대로 시도하여 처음 성공한 소스의 데이터를 반환"""
    priority = priority or SOURCE_PRIORITY
    errors = []
    for name in priority:
        fn = LOADERS.get(name)
        if fn is None:
            continue
        try:
            df = fn(symbol, start, end)
            if df is None or len(df) == 0:
                raise ValueError('빈 결과')
            if verbose:
                print('    [OK  ] {:<8} {:>6} rows  {} ~ {}'.format(
                    name, len(df), df['date'].min().date(), df['date'].max().date()))
            return df, name
        except Exception as e:
            errors.append((name, '{}: {}'.format(type(e).__name__, str(e))[:90]))
            if verbose:
                print('    [FAIL] {:<8} {}'.format(name, errors[-1][1]))
    raise RuntimeError('모든 소스 실패\n' + '\n'.join(
        ['  - {} : {}'.format(a, b) for a, b in errors]))


def load_all_indices(index_codes: Dict[str, str], start: str, end: str) -> pd.DataFrame:
    frames, used = [], {}
    for code, name in index_codes.items():
        print('  {} ({})'.format(code, name))
        df, src = load_price_multi(code, start, end)
        df['index_code'] = code
        df['index_name'] = name
        df['source'] = src
        used[code] = src
        frames.append(df)
        time.sleep(REQUEST_SLEEP)

    out = pd.concat(frames, ignore_index=True)
    out = out.sort_values(['index_code', 'date']).reset_index(drop=True)

    if any(s == 'fdr_etf' for s in used.values()):
        print()
        print('  *** 경고: fdr_etf(ETF 가격) 폴백이 사용되었습니다.')
        print('      지수 레벨이 아니므로 이격도·MDD 절대수치 해석에 주의하십시오.')
    return out


print('로더 정의 완료 :', list(LOADERS.keys()))

로더 정의 완료 : ['naver', 'yahoo', 'fdr', 'fdr_etf']


## 5. 수집 실행 & 검증

In [5]:
print('[1] 지수 수집 : {} ~ {}'.format(START_DATE, END_DATE))
df_index = load_all_indices(INDEX_CODES, START_DATE, END_DATE)

df_index['change_pct'] = df_index.groupby('index_code')['close'].pct_change() * 100.0

print()
print('=' * 90)
g = df_index.groupby(['index_code', 'index_name', 'source']).agg(
    rows=('date', 'size'), first=('date', 'min'), last=('date', 'max'),
    last_close=('close', 'last'))
print(g.to_string())

print()
print('PK 중복 :', df_index.duplicated(subset=['date', 'index_code']).sum())
na = df_index.isna().sum()
na = na[na > 0]
print('결측 :', '없음' if len(na) == 0 else '')
if len(na) > 0:
    print(na.to_string())

# OHLC 정합성
ok = df_index[['open', 'high', 'low', 'close']].notna().all(axis=1)
bad = df_index[ok & ((df_index['high'] < df_index[['open', 'close']].max(axis=1)) |
                     (df_index['low'] > df_index[['open', 'close']].min(axis=1)))]
zero_o = df_index[(df_index['open'].isna()) | (df_index['open'] <= 0)]
print('OHLC 역전 : {} rows | 시가 결측·0 : {} rows (GK 변동성에서 자동 제외)'.format(
    len(bad), len(zero_o)))

# 극단 변동일
ext = df_index[df_index['change_pct'].abs() > 7].sort_values('change_pct')
print()
print('일간 변동률 |7%| 초과 : {} rows (최근 10건)'.format(len(ext)))
if len(ext) > 0:
    e = ext.tail(10).copy()
    e['date'] = e['date'].dt.date
    print(e[['date', 'index_code', 'close', 'change_pct']].round(2).to_string(index=False))

[1] 지수 수집 : 2005-01-01 ~ 2026-07-29
  KS11 (KOSPI)
    [OK  ] naver      5322 rows  2005-01-03 ~ 2026-07-29
  KQ11 (KOSDAQ)
    [OK  ] naver      5322 rows  2005-01-03 ~ 2026-07-29

                              rows      first       last  last_close
index_code index_name source                                        
KQ11       KOSDAQ     naver   5322 2005-01-03 2026-07-29      662.68
KS11       KOSPI      naver   5322 2005-01-03 2026-07-29     5663.24

PK 중복 : 0
결측 : 
change_pct    2
OHLC 역전 : 2 rows | 시가 결측·0 : 2 rows (GK 변동성에서 자동 제외)

일간 변동률 |7%| 초과 : 50 rows (최근 10건)
      date index_code   close  change_pct
2026-06-09       KS11 8096.93        8.18
2020-03-24       KQ11  480.40        8.26
2026-05-21       KS11 7815.59        8.42
2026-04-01       KS11 5478.70        8.44
2020-03-24       KS11 1609.97        8.60
2020-03-20       KQ11  467.75        9.20
2026-03-05       KS11 5583.90        9.63
2008-10-30       KQ11  296.05       11.47
2008-10-30       KS11 1084.72       11.95
2

## 6. 분석 모듈 (1) 변동성

- **실현변동성**: 로그수익률 롤링 표준편차 × √252 (%)
- **EWMA**: `σ²ₜ = λσ²ₜ₋₁ + (1−λ)r²ₜ` (RiskMetrics, 평균 미차감)
- **Parkinson**: `σ² = E[ln(H/L)²] / (4ln2)`
- **Garman-Klass**: `σ² = 0.5·ln(H/L)² − (2ln2−1)·ln(C/O)²`
- **변동성 백분위**: EWMA의 롤링 백분위 → 국면 판정 임계값

범위기반(Parkinson·GK) 추정량은 시가 갭이 큰 날 종가기준과 괴리가 생깁니다. 대체재가 아니라 교차검증용입니다.


In [6]:
def rolling_pctile(s: pd.Series, window: int,
                   min_periods: Optional[int] = None) -> pd.Series:
    if min_periods is None:
        min_periods = max(60, window // 4)
    return s.rolling(window, min_periods=min_periods).apply(
        lambda x: float((x <= x[-1]).mean()), raw=True)


def add_returns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values('date').copy()
    df['ret_log'] = np.log(df['close']).diff()
    df['ret_pct'] = df['close'].pct_change() * 100.0
    return df


def add_volatility(df: pd.DataFrame) -> pd.DataFrame:
    r = df['ret_log']
    for w in VOL_WINDOWS:
        df['vol_{}d'.format(w)] = (r.rolling(w, min_periods=max(5, w // 2)).std(ddof=1)
                                   * np.sqrt(VOL_ANNUALIZE) * 100.0)

    var_ewma = (r.fillna(0.0) ** 2).ewm(alpha=1.0 - EWMA_LAMBDA, adjust=False).mean()
    df['vol_ewma'] = np.sqrt(var_ewma * VOL_ANNUALIZE) * 100.0

    ohlc_ok = (df[['open', 'high', 'low', 'close']] > 0).all(axis=1)
    hl = np.log(df['high'] / df['low']).where(ohlc_ok)
    co = np.log(df['close'] / df['open']).where(ohlc_ok)
    w, mp = VOL_RANGE_WINDOW, max(5, VOL_RANGE_WINDOW // 2)

    park_var = (hl ** 2) / (4.0 * np.log(2.0))
    df['vol_parkinson'] = np.sqrt(park_var.rolling(w, min_periods=mp).mean().clip(lower=0)
                                  * VOL_ANNUALIZE) * 100.0
    gk_var = 0.5 * hl ** 2 - (2.0 * np.log(2.0) - 1.0) * co ** 2
    df['vol_gk'] = np.sqrt(gk_var.rolling(w, min_periods=mp).mean().clip(lower=0)
                           * VOL_ANNUALIZE) * 100.0

    df['vol_pctile'] = rolling_pctile(df['vol_ewma'], VOL_PCTILE_WINDOW)
    return df


print('변동성 모듈 정의 완료')

변동성 모듈 정의 완료


## 7. 분석 모듈 (2) MDD

일별 낙폭·수중 경과일·러닝 MDD 와, 전고점 이탈~회복을 하나로 묶은 낙폭 국면 테이블을 산출합니다.
회복하지 못한 진행 중 국면은 `is_ongoing=True`, `days_to_recover=-1` 로 표시됩니다.


In [7]:
def add_drawdown(df: pd.DataFrame) -> pd.DataFrame:
    c = df['close']
    peak = c.cummax()
    df['peak'] = peak
    df['drawdown'] = (c / peak - 1.0) * 100.0
    df['mdd_running'] = df['drawdown'].cummin()
    under = (df['drawdown'] < 0).astype(int)
    grp = (under == 0).cumsum()
    df['dd_duration'] = under.groupby(grp).cumsum()
    return df


def drawdown_episodes(df: pd.DataFrame, top_n: int = 10,
                      min_depth: float = -5.0) -> pd.DataFrame:
    d = df.reset_index(drop=True)
    dd = d['drawdown'].values
    dates = pd.to_datetime(d['date']).values
    n = len(d)

    spans, start = [], None
    for i in range(n):
        if dd[i] < 0 and start is None:
            start = i
        elif dd[i] >= 0 and start is not None:
            spans.append((start, i))
            start = None
    if start is not None:
        spans.append((start, None))

    rows = []
    for s, e in spans:
        end = e if e is not None else n - 1
        t = s + int(np.argmin(dd[s:end + 1]))
        peak_i = max(s - 1, 0)
        depth = float(dd[t])
        if depth > min_depth:
            continue
        rows.append({
            'index_code': d['index_code'].iloc[0],
            'peak_date': pd.Timestamp(dates[peak_i]),
            'trough_date': pd.Timestamp(dates[t]),
            'recovery_date': pd.Timestamp(dates[e]) if e is not None else pd.NaT,
            'mdd_pct': depth,
            'peak_close': float(d['close'].iloc[peak_i]),
            'trough_close': float(d['close'].iloc[t]),
            'days_to_trough': int(t - peak_i),
            'days_to_recover': int(e - t) if e is not None else -1,
            'total_days': int(end - peak_i),
            'is_ongoing': bool(e is None),
        })

    out = pd.DataFrame(rows)
    if len(out) > 0:
        out = out.sort_values('mdd_pct').head(top_n).reset_index(drop=True)
    return out


print('MDD 모듈 정의 완료')

MDD 모듈 정의 완료


## 8. 분석 모듈 (3) 이격도

`이격도 = 종가 / MA(n) × 100`. 절대수치는 지수·기간마다 분포가 달라 판정은 롤링 z-score 기준으로 합니다.

| z-score | 구간 |
|---|---|
| ≥ +2 | OVERHEAT |
| +1 ~ +2 | STRONG |
| −1 ~ +1 | NEUTRAL |
| −2 ~ −1 | WEAK |
| ≤ −2 | OVERSOLD |


In [8]:
def add_disparity(df: pd.DataFrame) -> pd.DataFrame:
    for w in MA_WINDOWS:
        df['ma_{}'.format(w)] = df['close'].rolling(w, min_periods=max(2, w // 2)).mean()
    for w in DISPARITY_WINDOWS:
        df['disp_{}'.format(w)] = df['close'] / df['ma_{}'.format(w)] * 100.0

    base = df['disp_{}'.format(DISPARITY_Z_BASE)]
    mu = base.rolling(DISPARITY_Z_WINDOW, min_periods=60).mean()
    sd = base.rolling(DISPARITY_Z_WINDOW, min_periods=60).std(ddof=1)
    df['disp_z'] = (base - mu) / sd.replace(0, np.nan)
    df['disp_zone'] = pd.cut(df['disp_z'], bins=[-np.inf, -2, -1, 1, 2, np.inf],
                             labels=['OVERSOLD', 'WEAK', 'NEUTRAL', 'STRONG', 'OVERHEAT']
                             ).astype(object)
    return df


print('이격도 모듈 정의 완료')

이격도 모듈 정의 완료


## 9. 분석 모듈 (4) 국면전환

### 인과성 구분이 핵심입니다

- `regime_ms` : **smoothed** 확률 기반. 전표본을 조건부로 하므로 과거 국면 서술에는 적합하나 **백테스트 사용 금지**
- `regime_ms_filt` : **filtered** 확률 기반. 확률은 t 시점까지만 조건부 (파라미터는 전표본)
- `regime_rule` / `regime_rule_conf` : 추세(MA200) × 변동성 백분위. 완전 인과

### 휩쏘 억제 방식 변경 (v3 → v4)

v3의 `smooth_regime` 은 짧은 국면을 사후에 흡수하는 방식이라 **지속기간을 알아야 = 미래를 봐야** 했습니다.
v4는 `confirm_signal` 로 대체합니다. 신규 신호가 `CONFIRM_DAYS` 연속 유지될 때만 전환하며 과거 정보만 사용합니다.

국면 번호는 조건부 분산 오름차순으로 재정렬해 `0=저변동`, `k−1=고변동` 으로 고정합니다.
재정렬하지 않으면 실행할 때마다 라벨이 뒤바뀌어 DB 적재값과 충돌합니다.


In [9]:
def _ms_probs_2d(obj, nobs: int, k: int) -> Optional[np.ndarray]:
    a = np.asarray(obj)
    if a.ndim != 2:
        return None
    if a.shape == (k, nobs) and nobs != k:
        a = a.T
    return a if a.shape[0] == nobs else None


def _regime_order(p: np.ndarray, y: np.ndarray, k: int) -> List[int]:
    dm = y - y.mean()
    var_k = [float((p[:, i] * dm ** 2).sum() / max(p[:, i].sum(), 1e-12)) for i in range(k)]
    return list(np.argsort(var_k))


def confirm_signal(raw: pd.Series, confirm_days: int) -> pd.Series:
    """신규 신호가 confirm_days 연속 유지될 때만 전환 (과거 정보만 사용)"""
    if confirm_days is None or confirm_days <= 1:
        return raw.copy()
    def _na(x):
        return x is None or (isinstance(x, float) and np.isnan(x))

    v = list(raw.values)
    n = len(v)
    out = [np.nan] * n                       # 미확정 구간은 NaN 으로 통일 (dtype 일관성)
    state, streak_val, streak = np.nan, np.nan, 0

    for i in range(n):
        x = v[i]
        if _na(x):                           # 입력 결측: 직전 상태 유지, 연속카운트 초기화
            out[i] = state
            streak_val, streak = np.nan, 0
            continue

        if _na(streak_val) or x != streak_val:
            streak_val, streak = x, 1
        else:
            streak += 1

        if streak >= confirm_days and (_na(state) or x != state):
            state = x                        # confirm_days 연속 유지 시에만 전환
        out[i] = state

    return pd.Series(out, index=raw.index)


def add_regime_rule(df: pd.DataFrame) -> pd.DataFrame:
    ma_col = 'ma_{}'.format(REGIME_TREND_MA)
    if ma_col not in df.columns:
        df[ma_col] = df['close'].rolling(REGIME_TREND_MA,
                                         min_periods=REGIME_TREND_MA // 2).mean()
    up = df['close'] > df[ma_col]
    hv = df['vol_pctile'] > REGIME_VOL_PCTILE_TH
    lbl = np.where(up & ~hv, 'BULL_CALM',
                   np.where(up & hv, 'BULL_VOLATILE',
                            np.where(~up & ~hv, 'BEAR_CALM', 'BEAR_VOLATILE')))
    valid = df[ma_col].notna() & df['vol_pctile'].notna()
    df['regime_rule'] = pd.Series(lbl, index=df.index).where(valid)
    df['regime_rule_conf'] = confirm_signal(df['regime_rule'], CONFIRM_DAYS)
    return df


def regime_label(k) -> Optional[str]:
    if k is None or (isinstance(k, float) and np.isnan(k)):
        return np.nan
    k = int(k)
    return ['LOW_VOL', 'HIGH_VOL'][k] if MS_K_REGIMES == 2 else 'REGIME_{}'.format(k)


def add_regime_markov(df: pd.DataFrame) -> Tuple[pd.DataFrame, Optional[dict]]:
    """smoothed(서술용) 와 filtered(인과) 를 함께 산출"""
    for c in ['regime_ms', 'regime_ms_prob_high', 'regime_ms_filt', 'regime_ms_filt_prob_high']:
        df[c] = np.nan

    if (not RUN_MARKOV) or (not HAS_STATSMODELS):
        return df, None

    y = df['ret_pct'].dropna()
    if len(y) < MS_MIN_OBS:
        print('    [SKIP] 관측치 {} < MS_MIN_OBS {}'.format(len(y), MS_MIN_OBS))
        return df, None

    try:
        mod = MarkovRegression(y.values, k_regimes=MS_K_REGIMES, trend='c',
                               switching_variance=True)
        res = mod.fit(maxiter=MS_MAXITER, disp=False)
    except Exception as e:
        print('    [WARN] MS 추정 실패 -> 규칙기반만 사용 : {}'.format(e))
        return df, None

    p_sm = _ms_probs_2d(res.smoothed_marginal_probabilities, len(y), MS_K_REGIMES)
    p_fi = _ms_probs_2d(res.filtered_marginal_probabilities, len(y), MS_K_REGIMES)
    if p_sm is None:
        print('    [WARN] MS 확률 shape 불일치 -> 건너뜀')
        return df, None

    order = _regime_order(p_sm, y.values, MS_K_REGIMES)
    p_sm = p_sm[:, order]
    df.loc[y.index, 'regime_ms'] = np.argmax(p_sm, axis=1).astype(float)
    df.loc[y.index, 'regime_ms_prob_high'] = p_sm[:, -1]

    if p_fi is not None:
        p_fi = p_fi[:, order]
        df.loc[y.index, 'regime_ms_filt'] = np.argmax(p_fi, axis=1).astype(float)
        df.loc[y.index, 'regime_ms_filt_prob_high'] = p_fi[:, -1]

    df['regime_ms_lbl'] = df['regime_ms'].apply(regime_label)
    df['regime_ms_filt_lbl'] = df['regime_ms_filt'].apply(regime_label)
    df['regime_ms_filt_conf'] = confirm_signal(df['regime_ms_filt_lbl'], CONFIRM_DAYS)

    dm = y.values - y.values.mean()
    var_k = [float((p_sm[:, i] * dm ** 2).sum() / max(p_sm[:, i].sum(), 1e-12))
             for i in range(MS_K_REGIMES)]
    try:
        dur = [float(x) for x in np.asarray(res.expected_durations)[order]]
    except Exception:
        dur = [float('nan')] * MS_K_REGIMES

    info = {'nobs': int(len(y)),
            'ann_vol_by_regime': [round(float(np.sqrt(v * VOL_ANNUALIZE)), 2) for v in var_k],
            'expected_duration_days': [round(x, 1) for x in dur],
            'llf': round(float(res.llf), 2), 'aic': round(float(res.aic), 2)}
    return df, info


print('국면전환 모듈 정의 완료 (statsmodels: {})'.format('OK' if HAS_STATSMODELS else 'N/A'))

국면전환 모듈 정의 완료 (statsmodels: OK)


## 10. 분석 실행

In [10]:
def compute_analytics(df_all: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    outs, eps, infos = [], [], {}
    for code, g in df_all.groupby('index_code', sort=False):
        print('  - {} ({} rows)'.format(code, len(g)))
        g = add_returns(g)
        g = add_volatility(g)
        g = add_drawdown(g)
        g = add_disparity(g)
        g = add_regime_rule(g)
        g, info = add_regime_markov(g)
        infos[code] = info
        outs.append(g)
        e = drawdown_episodes(g, DD_TOP_N, DD_MIN_DEPTH_PCT)
        if len(e) > 0:
            eps.append(e)

    ana = pd.concat(outs, ignore_index=True).sort_values(['index_code', 'date'])
    ddf = pd.concat(eps, ignore_index=True) if eps else pd.DataFrame()
    return ana.reset_index(drop=True), ddf, infos


print('[2] 분석 지표 계산')
df_ana, df_dd, ms_info = compute_analytics(df_index)
print('-> analytics {} rows x {} cols | 낙폭국면 {} rows'.format(
    len(df_ana), df_ana.shape[1], len(df_dd)))

print()
print('[Markov Switching 추정 결과]')
for code, info in ms_info.items():
    if info is None:
        print('  {} : 미수행'.format(code))
    else:
        print('  {} : 국면별 연율변동성 {} % | 기대지속 {} 일 | AIC {}'.format(
            code, info['ann_vol_by_regime'], info['expected_duration_days'], info['aic']))

[2] 분석 지표 계산
  - KQ11 (5322 rows)
  - KS11 (5322 rows)
-> analytics 10644 rows x 43 cols | 낙폭국면 20 rows

[Markov Switching 추정 결과]
  KQ11 : 국면별 연율변동성 [14.49, 43.5] % | 기대지속 [29.0, 9.1] 일 | AIC 17760.9
  KS11 : 국면별 연율변동성 [14.2, 43.73] % | 기대지속 [99.1, 18.7] 일 | AIC 16224.35


## 11. 현재 상태 요약

In [11]:
def latest_summary(ana: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for code, g in ana.groupby('index_code', sort=False):
        g = g.sort_values('date')
        last = g.iloc[-1]
        pk = g.loc[g['close'] == last['peak'], 'date']
        rows.append({
            '지수': last['index_name'],
            '소스': last['source'],
            '기준일': last['date'].date(),
            '종가': round(float(last['close']), 2),
            '전일대비%': round(float(last['ret_pct']), 2) if pd.notna(last['ret_pct']) else np.nan,
            'vol_20d%': round(float(last['vol_20d']), 1) if pd.notna(last['vol_20d']) else np.nan,
            'vol_60d%': round(float(last['vol_60d']), 1) if pd.notna(last['vol_60d']) else np.nan,
            'EWMA%': round(float(last['vol_ewma']), 1),
            'Parkinson%': round(float(last['vol_parkinson']), 1)
                          if pd.notna(last['vol_parkinson']) else np.nan,
            'vol백분위': round(float(last['vol_pctile']), 3)
                        if pd.notna(last['vol_pctile']) else np.nan,
            '현재낙폭%': round(float(last['drawdown']), 2),
            '수중일수': int(last['dd_duration']),
            '전고점일': pk.iloc[-1].date() if len(pk) else None,
            '역대MDD%': round(float(g['drawdown'].min()), 2),
            '이격20': round(float(last['disp_{}'.format(DISPARITY_Z_BASE)]), 1),
            '이격z': round(float(last['disp_z']), 2) if pd.notna(last['disp_z']) else np.nan,
            '이격구간': last['disp_zone'],
            '규칙국면(확인)': last.get('regime_rule_conf'),
            'MS국면(filtered)': last.get('regime_ms_filt_lbl'),
            'MS고변동확률': round(float(last['regime_ms_filt_prob_high']), 3)
                            if pd.notna(last.get('regime_ms_filt_prob_high')) else np.nan,
        })
    return pd.DataFrame(rows)


summary = latest_summary(df_ana)
print('=' * 100)
print('현재 상태 요약')
print('=' * 100)
print(summary.T.to_string())

print()
print('=' * 100)
print('국면별 분포 및 평균 수익률 (규칙기반, 확인지연 적용)')
print('=' * 100)
for code, g in df_ana.groupby('index_code', sort=False):
    t = g.groupby('regime_rule_conf').agg(
        일수=('date', 'size'), 평균일간수익률=('ret_pct', 'mean'),
        평균변동성=('vol_20d', 'mean'), 평균낙폭=('drawdown', 'mean'))
    t['평균일간수익률'] = t['평균일간수익률'].round(4)
    t[['평균변동성', '평균낙폭']] = t[['평균변동성', '평균낙폭']].round(2)
    print('\n[{}]'.format(code))
    print(t.to_string())

print()
print('=' * 100)
print('상위 낙폭 국면 (깊이 {}% 이상)'.format(DD_MIN_DEPTH_PCT))
print('=' * 100)
if len(df_dd) > 0:
    show = df_dd.copy()
    for c in ['peak_date', 'trough_date', 'recovery_date']:
        show[c] = pd.to_datetime(show[c]).dt.date
    show['mdd_pct'] = show['mdd_pct'].round(2)
    print(show.to_string(index=False))

현재 상태 요약
                            0              1
지수                     KOSDAQ          KOSPI
소스                      naver          naver
기준일                2026-07-29     2026-07-29
종가                     662.68        5663.24
전일대비%                   -6.12          -5.98
vol_20d%                 67.6           82.2
vol_60d%                 60.2           73.9
EWMA%                    71.4           84.2
Parkinson%               56.5           69.7
vol백분위                  0.989            1.0
현재낙폭%                  -45.96         -37.87
수중일수                       62             26
전고점일               2026-04-27     2026-06-22
역대MDD%                 -68.46         -54.54
이격20                     83.4           79.9
이격z                     -2.93           -3.5
이격구간                 OVERSOLD       OVERSOLD
규칙국면(확인)        BEAR_VOLATILE  BULL_VOLATILE
MS국면(filtered)       HIGH_VOL       HIGH_VOL
MS고변동확률                   1.0            1.0

국면별 분포 및 평균 수익률 (규칙기반, 확인지연 적용)

[KQ11]
     

## 12. 국면 전환 이력

`regime_ms` (smoothed) 대비 `regime_ms_filt_conf` (filtered + 확인지연) 를 함께 봅니다.
두 전환 시점의 차이가 **실시간으로는 국면 전환을 얼마나 늦게 알 수 있는지**를 보여줍니다.


In [12]:
def regime_changes(ana: pd.DataFrame, col: str, last_n: int = 10) -> pd.DataFrame:
    rows = []
    for code, g in ana.groupby('index_code', sort=False):
        g = g.sort_values('date')
        s = g[col]
        chg = s.ne(s.shift()) & s.notna() & s.shift().notna()
        sub = g.loc[chg, ['date', 'index_code', 'close', col]].copy()
        sub['from'] = s.shift()[chg].values
        sub = sub.rename(columns={col: 'to'}).tail(last_n)
        rows.append(sub)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


for col, title in [('regime_rule_conf', '규칙기반(확인지연) — 인과'),
                   ('regime_ms_filt_conf', 'MS filtered(확인지연) — 인과'),
                   ('regime_ms_lbl', 'MS smoothed — 비인과, 서술용')]:
    if col not in df_ana.columns or df_ana[col].notna().sum() == 0:
        continue
    rc = regime_changes(df_ana, col)
    if len(rc) == 0:
        continue
    print('=' * 78)
    print('[{}]'.format(title))
    rc['date'] = rc['date'].dt.date
    rc['close'] = rc['close'].round(2)
    print(rc[['index_code', 'date', 'close', 'from', 'to']].to_string(index=False))
    print()

[규칙기반(확인지연) — 인과]
index_code       date   close          from            to
      KQ11 2025-03-05  746.95     BULL_CALM     BEAR_CALM
      KQ11 2025-04-09  643.39     BEAR_CALM BEAR_VOLATILE
      KQ11 2025-05-13  731.88 BEAR_VOLATILE     BEAR_CALM
      KQ11 2025-05-30  734.35     BEAR_CALM     BULL_CALM
      KQ11 2025-11-24  856.44     BULL_CALM BULL_VOLATILE
      KQ11 2025-12-11  934.64 BULL_VOLATILE     BULL_CALM
      KQ11 2026-01-28 1133.52     BULL_CALM BULL_VOLATILE
      KQ11 2026-06-10  951.63 BULL_VOLATILE BEAR_VOLATILE
      KQ11 2026-06-15 1034.03 BEAR_VOLATILE BULL_VOLATILE
      KQ11 2026-06-23  891.52 BULL_VOLATILE BEAR_VOLATILE
      KS11 2025-03-10 2570.39 BEAR_VOLATILE     BEAR_CALM
      KS11 2025-03-19 2628.62     BEAR_CALM     BULL_CALM
      KS11 2025-04-02 2505.86     BULL_CALM BEAR_VOLATILE
      KS11 2025-05-09 2577.27 BEAR_VOLATILE BULL_VOLATILE
      KS11 2025-07-03 3116.27 BULL_VOLATILE     BULL_CALM
      KS11 2025-07-08 3114.95     BULL_CALM BULL_VOLAT

## 13. CSV 저장

In [13]:
if SAVE_CSV:
    stamp = datetime.now().strftime('%Y%m%d')
    p1 = os.path.join(OUT_DIR, 'kr_index_daily_{}.csv'.format(stamp))
    df_index.to_csv(p1, index=False, encoding='utf-8-sig')
    print('saved :', p1)

    p2 = os.path.join(OUT_DIR, 'kr_index_analytics_{}.csv'.format(stamp))
    df_ana.to_csv(p2, index=False, encoding='utf-8-sig')
    print('saved :', p2)

    if len(df_dd) > 0:
        p3 = os.path.join(OUT_DIR, 'kr_index_drawdown_episode_{}.csv'.format(stamp))
        df_dd.to_csv(p3, index=False, encoding='utf-8-sig')
        print('saved :', p3)

    p4 = os.path.join(OUT_DIR, 'kr_index_summary_{}.csv'.format(stamp))
    summary.to_csv(p4, index=False, encoding='utf-8-sig')
    print('saved :', p4)
    print()
    print('백테스트 노트북 연결 : DATA_SOURCE = \'csv\' / CSV_PATH = r\'{}\''.format(p1))
else:
    print('SAVE_CSV = False -> 저장 생략')

saved : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\market_data\kr_index_daily_20260729.csv
saved : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\market_data\kr_index_analytics_20260729.csv
saved : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\market_data\kr_index_drawdown_episode_20260729.csv
saved : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\market_data\kr_index_summary_20260729.csv

백테스트 노트북 연결 : DATA_SOURCE = 'csv' / CSV_PATH = r'C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\market_data\kr_index_daily_20260729.csv'


## 14. MySQL 저장 (선택)

DDL은 DataFrame dtype 기준으로 **동적 생성**됩니다. 분석 파라미터를 바꿔 컬럼 구성이 달라지면
`CREATE TABLE IF NOT EXISTS` 로는 컬럼이 추가되지 않으므로 `RECREATE_ANALYTICS_TABLE=True` 로 1회 실행하십시오.


In [14]:
def _to_records(df: pd.DataFrame, cols: List[str]) -> List[dict]:
    tmp = df[cols].copy()
    for c in tmp.columns:
        if pd.api.types.is_datetime64_any_dtype(tmp[c]):
            tmp[c] = pd.to_datetime(tmp[c]).dt.date
    tmp = tmp.astype(object).where(pd.notnull(tmp), None)

    records = []
    for rec in tmp.to_dict('records'):
        clean = {}
        for k, v in rec.items():
            if isinstance(v, np.integer):
                v = int(v)
            elif isinstance(v, np.floating):
                v = None if np.isnan(v) else float(v)
            elif isinstance(v, np.bool_):
                v = bool(v)
            elif v is pd.NaT:
                v = None
            clean[k] = v
        records.append(clean)
    return records


def build_ddl(table: str, df: pd.DataFrame, pk: List[str]) -> str:
    """dtype 기반 DDL 동적 생성. np.issubdtype 은 StringDtype 에서 예외이므로 pandas API 사용"""
    lines = []
    for c in df.columns:
        s_ = df[c]
        if pd.api.types.is_datetime64_any_dtype(s_):
            t = 'DATE'
        elif c == 'index_code':
            t = 'VARCHAR(10)'
        elif pd.api.types.is_bool_dtype(s_):
            t = 'TINYINT(1)'
        elif pd.api.types.is_integer_dtype(s_):
            t = 'BIGINT'
        elif pd.api.types.is_float_dtype(s_):
            t = 'DOUBLE'
        else:
            t = 'VARCHAR(40)'
        lines.append('    `{}` {:<12} {}'.format(c, t, 'NOT NULL' if c in pk else 'NULL'))
    lines.append('    PRIMARY KEY ({})'.format(', '.join(['`{}`'.format(c) for c in pk])))
    return ('CREATE TABLE IF NOT EXISTS `{}` (\n{}\n) '
            'ENGINE=InnoDB DEFAULT CHARSET=utf8mb4').format(table, ',\n'.join(lines))


def upsert_dataframe(engine, table: str, df: pd.DataFrame,
                     cols: List[str], pk: List[str], chunk: int = 1000) -> int:
    from sqlalchemy import text
    col_list = ', '.join(['`{}`'.format(c) for c in cols])
    ph = ', '.join([':{}'.format(c) for c in cols])
    upd = ', '.join(['`{c}`=VALUES(`{c}`)'.format(c=c) for c in cols if c not in pk])
    sql = 'INSERT INTO `{}` ({}) VALUES ({}) ON DUPLICATE KEY UPDATE {}'.format(
        table, col_list, ph, upd)
    records = _to_records(df, cols)
    with engine.begin() as conn:
        for i in range(0, len(records), chunk):
            conn.execute(text(sql), records[i:i + chunk])
    return len(records)


if SAVE_DB:
    from sqlalchemy import text
    from stock_invest_function import get_engine

    engine = get_engine()
    idx_cols = ['date', 'index_code', 'index_name', 'open', 'high', 'low',
                'close', 'volume', 'change_pct', 'source']
    with engine.begin() as conn:
        conn.execute(text(build_ddl(TABLE_INDEX, df_index[idx_cols], ['date', 'index_code'])))
    print('{:<32} {} rows'.format(
        TABLE_INDEX,
        upsert_dataframe(engine, TABLE_INDEX, df_index, idx_cols,
                         ['date', 'index_code'], DB_CHUNK)))

    ana_cols = [c for c in df_ana.columns if c not in ('index_name', 'source')]
    with engine.begin() as conn:
        if RECREATE_ANALYTICS_TABLE:
            conn.execute(text('DROP TABLE IF EXISTS `{}`'.format(TABLE_ANALYTICS)))
            print('[INFO] {} DROP'.format(TABLE_ANALYTICS))
        conn.execute(text(build_ddl(TABLE_ANALYTICS, df_ana[ana_cols], ['date', 'index_code'])))
    print('{:<32} {} rows'.format(
        TABLE_ANALYTICS,
        upsert_dataframe(engine, TABLE_ANALYTICS, df_ana, ana_cols,
                         ['date', 'index_code'], DB_CHUNK)))

    if len(df_dd) > 0:
        with engine.begin() as conn:
            if RECREATE_ANALYTICS_TABLE:
                conn.execute(text('DROP TABLE IF EXISTS `{}`'.format(TABLE_DRAWDOWN)))
            conn.execute(text(build_ddl(TABLE_DRAWDOWN, df_dd, ['index_code', 'peak_date'])))
        print('{:<32} {} rows'.format(
            TABLE_DRAWDOWN,
            upsert_dataframe(engine, TABLE_DRAWDOWN, df_dd, list(df_dd.columns),
                             ['index_code', 'peak_date'], DB_CHUNK)))
else:
    print('SAVE_DB = False -> DB 저장 생략')

SAVE_DB = False -> DB 저장 생략


## 15. (선택) 시각화

In [15]:
if PLOT:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    PLOT_CODE = list(INDEX_CODES.keys())[0]
    PLOT_FROM = '2018-01-01'

    g = df_ana[(df_ana['index_code'] == PLOT_CODE) &
               (df_ana['date'] >= pd.to_datetime(PLOT_FROM))].sort_values('date')

    fig, ax = plt.subplots(4, 1, figsize=(14, 13), sharex=True,
                           gridspec_kw={'height_ratios': [3, 2, 2, 1.2]})

    ax[0].plot(g['date'], g['close'], lw=1.2, color='black', label=PLOT_CODE)
    for w in [20, 60, REGIME_TREND_MA]:
        c = 'ma_{}'.format(w)
        if c in g.columns:
            ax[0].plot(g['date'], g[c], lw=0.9, alpha=0.8, label='MA{}'.format(w))
    if g['regime_ms_filt'].notna().any():
        hi = (g['regime_ms_filt'] == (MS_K_REGIMES - 1)).values
        ax[0].fill_between(g['date'], g['close'].min(), g['close'].max(),
                           where=hi, color='red', alpha=0.10)
        h, l = ax[0].get_legend_handles_labels()
        ax[0].legend(handles=h + [Patch(facecolor='red', alpha=0.10,
                                        label='MS 고변동(filtered)')],
                     loc='upper left', fontsize=9)
    else:
        ax[0].legend(loc='upper left', fontsize=9)
    ax[0].set_title('{} 가격 / 이동평균 / 국면'.format(PLOT_CODE))
    ax[0].grid(alpha=0.3)

    for c, lb in [('vol_20d', '실현 20d'), ('vol_60d', '실현 60d'),
                  ('vol_ewma', 'EWMA'), ('vol_parkinson', 'Parkinson')]:
        if c in g.columns:
            ax[1].plot(g['date'], g[c], lw=1.0, label=lb)
    ax[1].set_ylabel('연율 변동성 %')
    ax[1].legend(loc='upper left', fontsize=8, ncol=4)
    ax[1].grid(alpha=0.3)

    ax[2].fill_between(g['date'], g['drawdown'], 0, color='steelblue', alpha=0.6)
    ax[2].plot(g['date'], g['mdd_running'], color='darkred', lw=0.9, label='running MDD')
    ax[2].set_ylabel('낙폭 %')
    ax[2].legend(loc='lower left', fontsize=8)
    ax[2].grid(alpha=0.3)

    ax[3].plot(g['date'], g['disp_z'], lw=0.9, color='purple')
    for y_, ls in [(2, '--'), (1, ':'), (0, '-'), (-1, ':'), (-2, '--')]:
        ax[3].axhline(y_, color='gray', lw=0.7, ls=ls)
    ax[3].set_ylabel('이격도 z ({}d)'.format(DISPARITY_Z_BASE))
    ax[3].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('PLOT = False -> 시각화 생략')

PLOT = False -> 시각화 생략
